# RAG con LangChain y NVIDIA NIM

Este notebook implementa el mismo pipeline RAG del laboratorio anterior usando las abstracciones de LangChain y los endpoints alojados de NVIDIA NIM.

| Laboratorio manual | LangChain + NVIDIA NIM |
|---|---|
| `EmbeddingService` | `NVIDIAEmbeddings` |
| `semantic_chunker()` | `RecursiveCharacterTextSplitter` |
| `ingest_documents()` | `QdrantVectorStore.add_documents()` |
| `retrieve()` | `vectorstore.as_retriever()` |
| `GenerationService` | `ChatNVIDIA` |
| `ask()` | pipeline LCEL con `|` |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> NVIDIA ofrece una integración nativa para LangChain mediante `langchain-nvidia-ai-endpoints`; no hace falta descargar pesos ni ejecutar un servidor local.


In [6]:
# Dependencias del laboratorio (Google Colab)
%pip install -qU langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 21.8 MB/s eta 0:00:00


In [2]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

from dotenv import load_dotenv

load_dotenv()

# Orden: Secret de Colab → variable de entorno → entrada oculta.
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

# Las clases de LangChain leen esta variable automáticamente.
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## Datos — los mismos documentos del laboratorio manual

Si existe `data/company_documents.json`, se utilizará. En un Colab nuevo, el notebook crea documentos ficticios mínimos para que toda la práctica pueda ejecutarse sin archivos adicionales.


In [3]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: str | Path) -> List[Dict]:
    path = Path(path)
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


In [4]:
DOCS_PATH = Path("company_documents.json")
company_documents = load_documents(DOCS_PATH)

for doc in company_documents:
    metadata = doc.get("metadata", {})
    print(f"  [{doc['id']}] {doc['title']}")
    print(f"      dept={metadata.get('department', '—')}  type={metadata.get('type', '—')}")

test_questions = [
    "¿Cuál es el precio del plan Enterprise de CloudSync Pro?",
    "¿Cuántos días a la semana pueden trabajar remotamente los empleados?",
    "¿Qué pasa con los reembolsos mayores a $100?",
    "¿Qué ocurre durante la primera semana de onboarding?",
]

print(f"✅ {len(company_documents)} documentos disponibles")


  [product_001] CloudSync Pro — Plan Enterprise
      dept=product  type=pricing
  [policy_001] Política de Trabajo Remoto 2024
      dept=hr  type=policy
  [process_001] Proceso de Reembolso a Clientes
      dept=support  type=process
  [guide_001] Checklist de Onboarding para Nuevos Empleados
      dept=hr  type=guide
✅ 4 documentos disponibles


---
## Parte 1 — Embeddings con NVIDIA NIM

**En el laboratorio manual:** `EmbeddingService` llamaba directamente al endpoint.

**En LangChain:** `NVIDIAEmbeddings` expone dos operaciones importantes:
- `embed_documents(texts)`: representa los fragmentos que serán indexados.
- `embed_query(text)`: representa una consulta para recuperación.

La integración puede conectarse al catálogo alojado de NVIDIA o a un NIM desplegado por la organización.


In [7]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)

# Prueba rápida y detección dinámica de la dimensión.
sample_vecs = embeddings.embed_documents([
    "El gato duerme en el sofá",
    "El felino descansa en el mueble",
])
EMBEDDING_DIM = len(sample_vecs[0])
print(f"✅ Embeddings: {EMBEDDING_MODEL} — dimensión={EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Embeddings: nvidia/nemotron-3-embed-1b — dimensión=2048


---
## Parte 2 — LLM de generación

**En el laboratorio manual:** `GenerationService.generate()` ocultaba la llamada al endpoint.

**En LangChain:** `ChatNVIDIA` implementa la interfaz estándar `ChatModel`: recibe mensajes y devuelve un `AIMessage`. Por eso puede insertarse directamente en un pipeline LCEL.


In [9]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=700,
)

response = llm.invoke([HumanMessage(content="Responde únicamente: hola")])
print(f"✅ LLM: {GENERATION_MODEL}")
print("Prueba:", response.content)


✅ LLM: nvidia/nemotron-3-super-120b-a12b
Prueba: hola


---
## Parte 3 — Chunking

`RecursiveCharacterTextSplitter` intenta separar primero por párrafos, después por líneas, oraciones y espacios. LangChain representa cada fragmento mediante un `Document`, que contiene `page_content` y `metadata`.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ".", " "],
)

lc_documents: List[Document] = []
for doc in company_documents:
    for index, chunk in enumerate(text_splitter.split_text(doc["content"])):
        lc_documents.append(Document(
            page_content=chunk,
            metadata={
                "source_id": doc["id"],
                "title": doc["title"],
                "chunk_index": index,
                **doc.get("metadata", {}),
            },
        ))

print(f"✅ Chunking: {len(company_documents)} documentos → {len(lc_documents)} chunks")
for document in lc_documents[:3]:
    print(f"  [{document.metadata['title']}] {document.page_content[:70]}...")


✅ Chunking: 4 documentos → 5 chunks
  [CloudSync Pro — Plan Enterprise] CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración...
  [Política de Trabajo Remoto 2024] Vigente desde enero 2024. Los empleados pueden trabajar de forma remot...
  [Proceso de Reembolso a Clientes] Paso 1: El cliente envía solicitud de reembolso por el portal de sopor...


---
## Parte 4 — Vector store con Qdrant

`QdrantVectorStore` utiliza `NVIDIAEmbeddings` internamente para indexar los documentos. La dimensión se detectó en la prueba anterior, evitando fijar un número que dependa del modelo.

Qdrant se ejecuta **en memoria** para que la celda sea repetible en Colab. En producción puede sustituirse por una URL de Qdrant Server o Qdrant Cloud.


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "company_docs_langchain_nim"
qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(lc_documents)
print(f"✅ Qdrant: {len(ids)} chunks indexados en '{COLLECTION}'")


✅ Qdrant: 5 chunks indexados en 'company_docs_langchain_nim'


---
## Parte 5 — Retrieval

`vectorstore.as_retriever()` crea un componente invocable y reutilizable. Con `similarity_score_threshold`, Qdrant devuelve hasta `k` documentos siempre que superen el umbral configurado.


In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.3},
)

question = test_questions[0]
docs = retriever.invoke(question)
print(f"Pregunta: {question}")
print(f"Chunks recuperados: {len(docs)}")
for document in docs:
    print(f"  [{document.metadata.get('title', 'Sin título')}] {document.page_content[:80]}...")


Pregunta: ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
Chunks recuperados: 3
  [CloudSync Pro — Plan Enterprise] CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo...
  [Política de Trabajo Remoto 2024] Vigente desde enero 2024. Los empleados pueden trabajar de forma remota hasta 3 ...
  [Checklist de Onboarding para Nuevos Empleados] Día 1: Configuración de TI y acceso a sistemas. Día 2: Orientación departamental...


---
## Parte 6 — Pipeline RAG completo con LCEL

LCEL (*LangChain Expression Language*) conecta los componentes con `|`:

```text
pregunta ──► retriever ──► format_docs ──► prompt ──► ChatNVIDIA ──► texto
       └────────────────────────────────────▲
```

El modelo recibe tanto la pregunta original como el contexto recuperado.


In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Sin título')}]\n{doc.page_content}"
        for doc in docs
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente empresarial. Responde únicamente con información "
        "del contexto proporcionado. Si el contexto no contiene la respuesta, "
        "indica que no tienes esa información. Responde en español y de forma concisa.",
    ),
    ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("=" * 60)
print("RAG CON LANGCHAIN + NVIDIA NIM")
print("=" * 60)
for question in test_questions:
    answer = rag_chain.invoke(question)
    print(f"\n❓ {question}")
    print(f"💬 {answer.strip()}")


RAG CON LANGCHAIN + NVIDIA NIM

❓ ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
💬 El plan Enterprise de CloudSync Pro cuesta **$49 USD por mes por usuario** (con compromiso anual). También existe una opción mensual sin compromiso a $59 USD por usuario.

❓ ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
💬 Los empleados pueden trabajar de forma remota hasta **3 días por semana**.

❓ ¿Qué pasa con los reembolsos mayores a $100?
💬 Los reembolsos mayores a $100 requieren la aprobación del supervisor antes de poder procesarse.

❓ ¿Qué ocurre durante la primera semana de onboarding?
💬 Durante la primera semana de onboarding, el nuevo empleado completa los módulos de entrenamiento obligatorios, que incluyen temas de seguridad, cumplimiento y cultura.


---
## Parte 7 — Retornar respuesta y fuentes

En el laboratorio manual, `ask()` retornaba `(answer, chunks)`. Con LCEL se puede conservar el resultado del *retriever* y generar la respuesta a partir de esos mismos documentos, evitando ejecutar la recuperación dos veces.


In [14]:
from langchain_core.runnables import RunnableParallel

# Primera etapa: conserva la pregunta y recupera los documentos una sola vez.
setup_and_retrieval = RunnableParallel(
    question=RunnablePassthrough(),
    documents=retriever,
)

# Segunda etapa: usa esos documentos tanto para responder como para citar fuentes.
answer_from_documents = (
    {
        "context": lambda item: format_docs(item["documents"]),
        "question": lambda item: item["question"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_with_sources = setup_and_retrieval.assign(answer=answer_from_documents)
result = rag_with_sources.invoke(test_questions[0])

print("Respuesta:", result["answer"].strip())
print("Fuentes:")
for document in result["documents"]:
    print(" -", document.metadata.get("title", "Sin título"))


Respuesta: El plan Enterprise de CloudSync Pro cuesta **$49 por mes por usuario** (con compromiso anual). También existe una opción mensual sin compromiso a $59 por mes por usuario.
Fuentes:
 - CloudSync Pro — Plan Enterprise
 - Política de Trabajo Remoto 2024
 - Checklist de Onboarding para Nuevos Empleados


---
## Parte 8 — Tuning del retriever

El umbral depende del modelo de embeddings y del corpus; no es una constante universal. Conviene inspeccionar resultados y evaluarlo con preguntas representativas antes de fijarlo en producción.


In [15]:
retriever_strict = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)

rag_chain_strict = (
    {
        "context": retriever_strict | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "¿Cuál es la política de trabajo remoto?"
print("THRESHOLD ESTRICTO (0.5):")
print(rag_chain_strict.invoke(question).strip())


THRESHOLD ESTRICTO (0.5):
La política de trabajo remoto (vigente desde enero 2024) permite a los empleados trabajar de forma remota hasta **3 días por semana**, siempre que obtengan la aprobación de su jefe directo. Incluye un **estipendio de equipos de $500 anuales** para adecuar la oficina en casa, exige que las **videollamadas sean obligatorias en reuniones de equipo** y establece un **horario base de colaboración de 10:00 – 15:00 hora local** del empleado.
